In [ ]:
# Cell 0: GPU + deps
import subprocess, sys

pkgs = [
    "bitsandbytes>=0.46.1",
    "transformers>=4.47.0",
    "peft>=0.13.0",
    "trl>=0.12.0",
    "datasets>=3.0.0",
    "accelerate>=1.0.0",
    "safetensors>=0.4.0",
]
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs,
    capture_output=True, text=True
)
if r.returncode != 0:
    print("STDERR:", r.stderr[-1000:])
    raise RuntimeError("pip install failed")
print("Dependencies installed.", flush=True)

import torch
assert torch.cuda.is_available(), "No GPU — abort."
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB", flush=True)


In [ ]:
# Cell 1: Clone repo
import subprocess, os, sys

REPO    = "https://github.com/Dyly-666/testing-local-ai-1.git"
WORKDIR = "/kaggle/working/project"

if os.path.exists(WORKDIR):
    r = subprocess.run(["git", "-C", WORKDIR, "pull"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", REPO, WORKDIR], capture_output=True, text=True)

if r.returncode != 0:
    print("STDOUT:", r.stdout)
    print("STDERR:", r.stderr)
    raise RuntimeError("git clone/pull failed")

os.chdir(WORKDIR)
print("Working dir:", os.getcwd(), flush=True)
print("Files:", sorted(os.listdir(".")), flush=True)


In [ ]:
# Cell 2: Train QLoRA adapter — Qwen3-8B, rank=16, 3 epochs
import subprocess, sys, os, json, pathlib
os.chdir("/kaggle/working/project")

r = subprocess.run(
    [sys.executable, "-m", "scripts.train_adapter",
     "--config", "configs/train.qwen3-8b.json",
     "--allow-pending-review",
     "--resume-from-checkpoint", "none"],
    capture_output=True, text=True
)
print(r.stdout[-5000:] or "(no stdout)")
if r.returncode != 0:
    print("STDERR:", r.stderr[-3000:])
    raise RuntimeError("Training FAILED")

# Verify adapter saved
adapter = pathlib.Path("/kaggle/working/adapter_v24/adapter_model.safetensors")
assert adapter.exists() and adapter.stat().st_size > 1_000_000, \
    f"Adapter save failed — size={adapter.stat().st_size if adapter.exists() else 'missing'}"
print(f"Adapter saved: {adapter.stat().st_size/1e6:.1f} MB")
